# Nghiên cứu TF-IDF + Linear SVM trên ViFN


**Notebook 02 — ViFN Experiments, bản hoàn chỉnh và đã đồng bộ artifact**

- **Bài toán:** phân loại `Real (0)` và `Fake (1)` trên văn bản tiếng Việt.
- **Dữ liệu huấn luyện cuối:** ViFN V1 đã loại exact duplicate và exact Train–Test overlap.
- **Mô hình cuối:** word TF-IDF `(1, 2)` + `LinearSVC(C=1.0, class_weight="balanced")`.
- **Nguồn sự thật:** `reports/vifn_v1/`, `models/vifn_v1/` và Official Test gốc.
- **Nguyên tắc:** Test không tham gia feature selection, tuning hoặc learning curve.

Mỗi header nằm trong một markdown cell riêng; nội dung, code và kết quả được tách thành các cell độc lập.


In [ ]:
from pathlib import Path
import hashlib
import json
import unicodedata

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data").is_dir()
        and (path / "reports").is_dir()
        and (path / "scripts").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc project.")

RAW_DIR = ROOT / "data/raw/vifn_v1"
DATA_DIR = ROOT / "data/processed/vifn_v1"
REPORT_DIR = ROOT / "reports/vifn_v1"
FINAL_DIR = REPORT_DIR / "final"
MODEL_PATH = ROOT / "models/vifn_v1/tuned_word_bigram.joblib"
LABEL_NAMES = {0: "Real", 1: "Fake"}

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split()).strip()

print("Project root:", ROOT)
print("Raw data:", RAW_DIR)
print("Reports:", REPORT_DIR)


## 1. Introduction


ViFN là bộ dữ liệu phân loại tin thật/tin giả tiếng Việt. Nghiên cứu sử dụng TF-IDF và Linear SVM vì đây là baseline nhanh, phù hợp ma trận văn bản thưa và cho phép xem trực tiếp trọng số đặc trưng.

Mục tiêu:

1. audit duplicate và Train–Test overlap trước khi huấn luyện;
2. so sánh sáu cách biểu diễn TF-IDF bằng group-aware CV;
3. chọn `C` và `class_weight` theo Macro F1;
4. đánh giá model đóng băng trên Official Test;
5. phân tích lỗi, hệ số đặc trưng, source bias và learning curve.

**Script tổng quát:** [`04_run_vifn_cv.py`](../scripts/02_vifn_experiments/04_run_vifn_cv.py), [`05_tune_vifn_svm.py`](../scripts/02_vifn_experiments/05_tune_vifn_svm.py), [`06_final_evaluate_vifn.py`](../scripts/02_vifn_experiments/06_final_evaluate_vifn.py).


## 2. Dataset Exploration


Dữ liệu gốc nằm trong `data/raw/vifn_v1/`:

- `train_data.csv`: 1.124 mẫu, gồm 559 Real và 565 Fake;
- `test_data.csv`: 282 mẫu Official Test, gồm 144 Real và 138 Fake;
- trường văn bản: `content`;
- trường nhãn: `label`.

[`01_inspect_vifn.py`](../scripts/02_vifn_experiments/01_inspect_vifn.py) khảo sát phân bố nhãn, missing, exact duplicate, nhóm mâu thuẫn nhãn và độ dài. [`02_audit_vifn.py`](../scripts/02_vifn_experiments/02_audit_vifn.py) kiểm tra exact/near duplicate trong Train và giữa Train–Test mà không đọc nhãn Test trong giai đoạn audit.


In [ ]:
raw_train = pd.read_csv(RAW_DIR / "train_data.csv")
raw_test = pd.read_csv(RAW_DIR / "test_data.csv")
clean_train = pd.read_csv(DATA_DIR / "train.csv")

dataset_summary = pd.DataFrame([
    {
        "dataset": "Raw Train",
        "samples": len(raw_train),
        "real_0": int((raw_train["label"] == 0).sum()),
        "fake_1": int((raw_train["label"] == 1).sum()),
        "fake_rate": float((raw_train["label"] == 1).mean()),
        "text_column": "content",
    },
    {
        "dataset": "Clean Train",
        "samples": len(clean_train),
        "real_0": int((clean_train["label"] == 0).sum()),
        "fake_1": int((clean_train["label"] == 1).sum()),
        "fake_rate": float((clean_train["label"] == 1).mean()),
        "text_column": "text",
    },
    {
        "dataset": "Official Test",
        "samples": len(raw_test),
        "real_0": int((raw_test["label"] == 0).sum()),
        "fake_1": int((raw_test["label"] == 1).sum()),
        "fake_rate": float((raw_test["label"] == 1).mean()),
        "text_column": "content",
    },
])
display(dataset_summary)

assert dataset_summary["samples"].tolist() == [1124, 924, 282]
assert raw_train["content"].notna().all()
assert raw_test["content"].notna().all()


### 2.1. Snapshot dữ liệu


| Tập | Số mẫu | Real | Fake | Tỷ lệ Fake |
|---|---:|---:|---:|---:|
| Raw Train | 1.124 | 559 | 565 | 50,27% |
| Clean Train | 924 | 559 | 365 | 39,50% |
| Official Test | 282 | 144 | 138 | 48,94% |

Việc loại duplicate/overlap làm giảm chủ yếu lớp Fake trong Train. Đây là thay đổi phân bố cần được ghi nhận khi giải thích `class_weight="balanced"` và khả năng tổng quát.


## 3. Data Preprocessing


[`03_prepare_vifn.py`](../scripts/02_vifn_experiments/03_prepare_vifn.py) thực hiện:

1. chuẩn hóa Unicode về NFC;
2. gộp khoảng trắng và bỏ khoảng trắng đầu/cuối;
3. loại exact duplicate trong Train;
4. loại khỏi Train mọi văn bản trùng exact với Test;
5. giữ Official Test nguyên vẹn;
6. lưu metadata làm sạch để audit.

Kết quả: 136 duplicate rows bị loại, sau đó loại thêm 64 Train texts trùng Test; Clean Train còn 924 mẫu. Test có 9 duplicate nội bộ nhưng không bị sửa để giữ đúng giao thức Official Test.


In [ ]:
with (DATA_DIR / "cleaning_metadata.json").open(encoding="utf-8") as file:
    cleaning = json.load(file)

display(pd.Series(cleaning, name="value").to_frame())

normalized_test = raw_test["content"].map(normalize_text)
assert len(clean_train) == cleaning["final_train"] == 924
assert clean_train["text"].is_unique
assert set(clean_train["text"]).isdisjoint(set(normalized_test))
assert cleaning["test_modified"] is False
assert cleaning["test_internal_duplicates"] == 9


### 3.1. Near-duplicate và leakage


[`02_audit_vifn.py`](../scripts/02_vifn_experiments/02_audit_vifn.py) từng ghi nhận 67 Official Test rows có nearest Train similarity từ 0,95 trước khi làm sạch. `03_prepare_vifn.py` loại exact overlap khỏi Train; [`04_run_vifn_cv.py`](../scripts/02_vifn_experiments/04_run_vifn_cv.py) tiếp tục kiểm tra character TF-IDF cosine similarity và dừng nếu còn Train–Test near-duplicate từ 0,95.

Trong Train, near-duplicate được gom bằng Union-Find và truyền `group_id` vào `StratifiedGroupKFold`. Test labels không được dùng trong các bước audit/lựa chọn model.


In [ ]:
near_duplicate_audit = pd.read_csv(REPORT_DIR / "train_test_near_duplicates.csv")
print("Suspect pairs recorded before cleaning:", len(near_duplicate_audit))
print("Minimum recorded similarity:", near_duplicate_audit["similarity"].min())
assert len(near_duplicate_audit) == 67
assert near_duplicate_audit["similarity"].ge(0.95).all()

with (FINAL_DIR / "test_metrics.json").open(encoding="utf-8") as file:
    final_metadata = json.load(file)

test_sha256 = hashlib.sha256((RAW_DIR / "test_data.csv").read_bytes()).hexdigest()
assert test_sha256 == final_metadata["test_sha256"]
print("Official Test SHA256:", test_sha256)


## 4. Feature Extraction


[`04_run_vifn_cv.py`](../scripts/02_vifn_experiments/04_run_vifn_cv.py) so sánh sáu cấu hình trên cùng 5-fold group-aware CV:

| ID | Đặc trưng | Cân bằng lớp |
|---|---|---|
| M01 | word unigram | Không |
| M02 | word unigram | Có |
| M03 | word unigram–bigram | Có |
| M04 | word bigram sau tách từ | Có |
| M05 | character `char_wb (3,5)` | Có |
| M06 | word + character | Có |

Các cấu hình word dùng `lowercase=True`, `min_df=2` và `sublinear_tf=True`.


In [ ]:
feature_results = pd.read_csv(REPORT_DIR / "fixed_experiments_cv.csv")
feature_view = feature_results[
    [
        "experiment", "mean_accuracy", "mean_macro_f1", "std_macro_f1",
        "mean_precision_1", "mean_recall_1", "mean_f1_1",
    ]
].sort_values("mean_macro_f1", ascending=False)
display(feature_view.reset_index(drop=True))

best_feature = feature_view.iloc[0]
assert best_feature["experiment"] == "M03_word_bigram"
assert np.isclose(best_feature["mean_macro_f1"], 0.8357943637955956)

ax = feature_view.set_index("experiment")[
    ["mean_macro_f1", "mean_precision_1", "mean_recall_1"]
].plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=25)
ax.set_title("ViFN: 5-fold group-aware CV")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()


### 4.1. Kết quả feature


| Cấu hình | CV Accuracy | CV Macro F1 | Precision Fake | Recall Fake |
|---|---:|---:|---:|---:|
| M03 word bigram | **0,8452** | **0,8358 ± 0,0149** | **0,8298** | 0,7671 |
| M04 segmented | 0,8409 | 0,8312 | 0,8230 | 0,7616 |
| M06 word + char | 0,8257 | 0,8155 | 0,8001 | 0,7479 |
| M05 char | 0,8106 | 0,8009 | 0,7699 | 0,7479 |
| M01 unigram | 0,8074 | 0,7932 | 0,7940 | 0,6932 |
| M02 unigram balanced | 0,7987 | 0,7875 | 0,7575 | 0,7233 |

Word unigram–bigram đạt Macro F1 cao nhất. Segmentation gần tương đương nhưng không vượt M03; kết hợp character không cải thiện trong dữ liệu này.


## 5. Model Training


Model đã chọn được refit trên toàn bộ 924 Clean Train samples và lưu tại `models/vifn_v1/tuned_word_bigram.joblib`:

~~~python
Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2),
        lowercase=True, min_df=2, sublinear_tf=True,
    )),
    ("svm", LinearSVC(
        C=1.0, class_weight="balanced",
        random_state=42, max_iter=5000,
    )),
])
~~~

Script: [`05_tune_vifn_svm.py`](../scripts/02_vifn_experiments/05_tune_vifn_svm.py).


In [ ]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Thiếu model: {MODEL_PATH}")

final_model = joblib.load(MODEL_PATH)
tfidf = final_model.named_steps["tfidf"]
svm = final_model.named_steps["svm"]

model_config = pd.Series({
    "analyzer": tfidf.analyzer,
    "ngram_range": tfidf.ngram_range,
    "min_df": tfidf.min_df,
    "sublinear_tf": tfidf.sublinear_tf,
    "vocabulary_size": len(tfidf.vocabulary_),
    "C": svm.C,
    "class_weight": svm.class_weight,
    "max_iter": svm.max_iter,
    "train_samples": len(clean_train),
})
display(model_config.to_frame("value"))

assert tfidf.ngram_range == (1, 2)
assert len(tfidf.vocabulary_) == 10845
assert svm.C == 1.0 and svm.class_weight == "balanced"


## 6. Hyperparameter Tuning


GridSearchCV thử:

- `C ∈ {0.01, 0.1, 1, 10, 100}`;
- `class_weight ∈ {None, "balanced"}`;
- 5-fold `StratifiedGroupKFold`;
- `refit="macro_f1"`.

Việc tính Accuracy, Precision, Recall và F1 song song giúp đọc trade-off, nhưng chỉ Macro F1 quyết định hạng model.


In [ ]:
tuning = pd.read_csv(REPORT_DIR / "svm_tuning_cv.csv")
tuning_view = tuning.sort_values(["rank_test_macro_f1", "param_svm__C"])
display(tuning_view.reset_index(drop=True))

best_cv = tuning.sort_values("rank_test_macro_f1").iloc[0]
assert best_cv["param_svm__C"] == 1.0
assert best_cv["param_svm__class_weight"] == "balanced"
assert np.isclose(best_cv["mean_test_macro_f1"], 0.8357943637955956)


### 6.1. Kết luận tuning


`C=1.0, class_weight="balanced"` đứng hạng 1 với CV Macro F1 **0,8358 ± 0,0149**. Cùng `C=1.0` nhưng không cân bằng đạt **0,8232**, Precision Fake cao hơn nhưng Recall Fake thấp hơn.

Train Macro F1 của cấu hình thắng là **0,9997**, cao hơn đáng kể CV score. Learning curve ở phần 10 được dùng để đánh giá mức overfitting này.


## 7. Model Evaluation


[`06_final_evaluate_vifn.py`](../scripts/02_vifn_experiments/06_final_evaluate_vifn.py) load model đã đóng băng và đánh giá một lần trên Official Test 282 mẫu. Script có guard chống ghi đè kết quả.

Ngoài metric chính trên toàn Official Test, script báo cáo phân tích phụ trên 273 unique Test texts. Official Test không bị sửa, vì loại duplicate sau khi xem Test sẽ làm thay đổi benchmark.


In [ ]:
predictions = pd.read_csv(FINAL_DIR / "test_predictions.csv")
assert len(predictions) == len(raw_test) == 282
assert np.array_equal(predictions["actual_label"].to_numpy(), raw_test["label"].to_numpy())

test_text = raw_test["content"].map(normalize_text)
model_predictions = final_model.predict(test_text)
assert np.array_equal(model_predictions, predictions["predicted_label"].to_numpy())

def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision_fake": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_fake": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_fake": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

official_recomputed = calculate_metrics(
    predictions["actual_label"], predictions["predicted_label"]
)
official_recorded = final_metadata["official_test_metrics"]

metric_comparison = pd.DataFrame({
    "recorded": official_recorded,
    "recomputed": official_recomputed,
})
metric_comparison["absolute_difference"] = (
    metric_comparison["recorded"].astype(float)
    - metric_comparison["recomputed"].astype(float)
).abs()
display(metric_comparison)
assert metric_comparison["absolute_difference"].max() < 1e-12

ConfusionMatrixDisplay(
    np.array([
        [official_recomputed["tn"], official_recomputed["fp"]],
        [official_recomputed["fn"], official_recomputed["tp"]],
    ]),
    display_labels=["Real", "Fake"],
).plot(cmap="Blues")
plt.title("ViFN Official Test confusion matrix")
plt.show()


### 7.1. Official Test snapshot


| Metric | Official Test |
|---|---:|
| Accuracy | **0,8794** |
| Macro F1 | **0,8788** |
| Precision Fake | **0,9194** |
| Recall Fake | **0,8261** |
| F1 Fake | **0,8702** |
| TN / FP / FN / TP | **134 / 10 / 24 / 114** |

Unique-only analysis có 273 mẫu, Macro F1 **0,8778** và F1 Fake **0,8653**. Kết quả gần Official Test, nên 9 duplicate nội bộ không thay đổi kết luận chính.


In [ ]:
eval_frame = pd.DataFrame({
    "text": test_text,
    "actual": predictions["actual_label"],
    "predicted": predictions["predicted_label"],
})
unique_eval = eval_frame.drop_duplicates("text", keep="first")
unique_recomputed = calculate_metrics(unique_eval["actual"], unique_eval["predicted"])

display(pd.DataFrame({
    "recorded": final_metadata["unique_test_metrics"],
    "recomputed": unique_recomputed,
}))
assert len(unique_eval) == final_metadata["unique_test_samples"] == 273
for key, value in final_metadata["unique_test_metrics"].items():
    assert np.isclose(float(value), float(unique_recomputed[key]))


## 8. Error Analysis


[`07_analyze_vifn_errors.py`](../scripts/02_vifn_experiments/07_analyze_vifn_errors.py) nối prediction với văn bản và SVM decision score. Kết quả có **34 lỗi**: 10 False Positive và 24 False Negative.

Decision score dương nghiêng về Fake; score âm nghiêng về Real. `abs(score)` lớn ở một dự đoán sai cho biết model sai với margin mạnh, hữu ích để ưu tiên review.


In [ ]:
error_analysis = pd.read_csv(FINAL_DIR / "error_analysis.csv")
error_by_length = pd.read_csv(FINAL_DIR / "error_by_length.csv")
display(error_analysis["error_type"].value_counts().to_frame("count"))
display(error_by_length)

assert len(error_analysis) == 34
assert (error_analysis["error_type"] == "FP").sum() == 10
assert (error_analysis["error_type"] == "FN").sum() == 24
assert int(error_by_length["errors"].sum()) == 34

top_errors = error_analysis.nlargest(6, "margin_magnitude").copy()
top_errors["text_preview"] = top_errors["text"].str.slice(0, 180)
display(top_errors[
    ["error_type", "decision_score", "length", "text_preview"]
])


### 8.1. Lỗi theo độ dài


| Độ dài | Số mẫu | Số lỗi | Error rate |
|---|---:|---:|---:|
| 1–80 | 66 | 9 | 13,64% |
| 81–150 | 78 | 12 | 15,38% |
| 151–350 | 81 | 10 | 12,35% |
| >350 | 57 | 3 | 5,26% |

Trong Test này, nhóm dài hơn 350 ký tự có error rate thấp nhất. Đây là quan sát mô tả, không chứng minh độ dài là nguyên nhân.


## 9. Model Interpretability


[`08_explain_vifn_svm.py`](../scripts/02_vifn_experiments/08_explain_vifn_svm.py) xuất:

- 30 n-gram có hệ số dương lớn nhất, nghiêng về Fake;
- 30 n-gram có hệ số âm lớn nhất, nghiêng về Real;
- đóng góp cục bộ của feature cho 34 dự đoán sai.

Hệ số mô tả hành vi model trong dataset, không phải bằng chứng rằng một từ làm nội dung trở thành thật hoặc giả.


In [ ]:
global_features = pd.read_csv(REPORT_DIR / "interpretability/global_features.csv")
error_explanations = pd.read_csv(REPORT_DIR / "interpretability/error_explanations.csv")
display(global_features.groupby("class_direction").head(15))
print("Explained errors:", len(error_explanations))

assert len(global_features) == 60
assert len(error_explanations) == 34

plot_features = global_features.groupby("class_direction").head(15)
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, direction, color in [
    (axes[0], "Real", "tab:blue"),
    (axes[1], "Fake", "tab:orange"),
]:
    subset = plot_features[plot_features["class_direction"] == direction]
    subset.sort_values("coefficient").plot.barh(
        x="feature", y="coefficient", ax=ax,
        legend=False, color=color, title=f"Nghiêng về {direction}",
    )
plt.tight_layout()
plt.show()


## 10. Additional Experiments


### 10.1. Source-bias ablation


[`09_audit_vifn_source_bias.py`](../scripts/02_vifn_experiments/09_audit_vifn_source_bias.py) kiểm tra các marker nguồn `vnexpress`, `vtcnews`, `socialnews` và `hoithocuocsong`. Sau khi che các marker này và chạy lại cùng CV, Macro F1 giảm từ **0,8358** xuống **0,8238**.

Kết quả cho thấy marker nguồn đóng góp một phần tín hiệu, nhưng loại chúng không làm model mất toàn bộ khả năng. Đây vẫn là cảnh báo về source bias và khả năng chuyển miền.


In [ ]:
source_ablation = pd.read_csv(REPORT_DIR / "source_bias/source_ablation_cv.csv")
display(source_ablation)
source_delta = (
    source_ablation.loc[
        source_ablation["experiment"] == "source_markers_removed", "macro_f1_mean"
    ].iloc[0]
    - source_ablation.loc[
        source_ablation["experiment"] == "original", "macro_f1_mean"
    ].iloc[0]
)
print("Macro F1 change after masking source markers:", f"{source_delta:+.4f}")
assert np.isclose(source_delta, -0.0119604409864816)


### 10.2. Learning curve


[`10_learning_curve_vifn.py`](../scripts/02_vifn_experiments/10_learning_curve_vifn.py) dùng cấu hình đóng băng và chỉ truy cập Clean Train. CV Macro F1 tăng khi thêm dữ liệu, trong khi Train Macro F1 gần 1,0 ở mọi kích thước.


In [ ]:
curve = pd.read_csv(REPORT_DIR / "learning_curve/learning_curve.csv")
display(curve)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve["train_size"], curve["train_macro_f1_mean"], marker="o", label="Train")
ax.plot(curve["train_size"], curve["cv_macro_f1_mean"], marker="o", label="CV")
ax.fill_between(
    curve["train_size"],
    curve["cv_macro_f1_mean"] - curve["cv_macro_f1_std"],
    curve["cv_macro_f1_mean"] + curve["cv_macro_f1_std"],
    alpha=0.15,
)
ax.set(xlabel="Training samples per fold", ylabel="Macro F1", ylim=(0, 1.05))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

assert np.isclose(curve.iloc[-1]["cv_macro_f1_mean"], 0.8357943637955956)
print("Final generalization gap:", f'{curve.iloc[-1]["generalization_gap"]:.4f}')


Tại 739 training samples/fold, CV Macro F1 đạt **0,8358 ± 0,0149**, nhưng generalization gap vẫn là **0,1639**. Thêm dữ liệu đa dạng và giảm tín hiệu đặc thù nguồn có thể hữu ích hơn việc tăng độ phức tạp model.


## 11. Discussion


### 11.1. Kết luận chính


- Word unigram–bigram là feature tốt nhất theo group-aware CV.
- `C=1.0, class_weight="balanced"` đạt CV Macro F1 **0,8358**.
- Official Test Macro F1 đạt **0,8788**, F1 Fake **0,8702**.
- Model có Recall Fake thấp hơn Precision Fake: 24 Fake bị bỏ sót, so với 10 Real bị cảnh báo nhầm.
- Learning curve và source ablation cho thấy overfitting/source bias vẫn đáng lưu ý.


### 11.2. Threats to validity


1. Clean Train giảm mạnh số Fake do duplicate/overlap, làm phân bố thay đổi.
2. Test có 9 duplicate nội bộ; Official score vẫn giữ nguyên benchmark, unique score chỉ là phân tích phụ.
3. Near-duplicate threshold 0,95 không phát hiện mọi paraphrase.
4. Hệ số từ vựng có thể học nguồn hoặc chủ đề thay vì độ tin cậy.
5. Score in-domain cao không bảo đảm chuyển tốt sang ReINTEL hoặc dữ liệu mới.


## Phụ lục A — Scripts và thứ tự chạy


| Bước | Script | Artifact chính |
|---:|---|---|
| 1 | [`01_inspect_vifn.py`](../scripts/02_vifn_experiments/01_inspect_vifn.py) | EDA log |
| 2 | [`02_audit_vifn.py`](../scripts/02_vifn_experiments/02_audit_vifn.py) | `train_test_near_duplicates.csv` |
| 3 | [`03_prepare_vifn.py`](../scripts/02_vifn_experiments/03_prepare_vifn.py) | Clean Train, cleaning metadata |
| 4 | [`04_run_vifn_cv.py`](../scripts/02_vifn_experiments/04_run_vifn_cv.py) | Feature CV |
| 5 | [`05_tune_vifn_svm.py`](../scripts/02_vifn_experiments/05_tune_vifn_svm.py) | Tuning CV, frozen model |
| 6 | [`06_final_evaluate_vifn.py`](../scripts/02_vifn_experiments/06_final_evaluate_vifn.py) | Final Test metrics/predictions |
| 7 | [`07_analyze_vifn_errors.py`](../scripts/02_vifn_experiments/07_analyze_vifn_errors.py) | Error analysis |
| 8 | [`08_explain_vifn_svm.py`](../scripts/02_vifn_experiments/08_explain_vifn_svm.py) | Global/local explanations |
| 9 | [`09_audit_vifn_source_bias.py`](../scripts/02_vifn_experiments/09_audit_vifn_source_bias.py) | Source ablation |
| 10 | [`10_learning_curve_vifn.py`](../scripts/02_vifn_experiments/10_learning_curve_vifn.py) | Learning curve |

~~~bash
python scripts/02_vifn_experiments/01_inspect_vifn.py
python scripts/02_vifn_experiments/02_audit_vifn.py
python scripts/02_vifn_experiments/03_prepare_vifn.py
python scripts/02_vifn_experiments/04_run_vifn_cv.py
python scripts/02_vifn_experiments/05_tune_vifn_svm.py
# Chỉ chạy Final Test khi chưa có test_metrics.json:
python scripts/02_vifn_experiments/06_final_evaluate_vifn.py
python scripts/02_vifn_experiments/07_analyze_vifn_errors.py
python scripts/02_vifn_experiments/08_explain_vifn_svm.py
python scripts/02_vifn_experiments/09_audit_vifn_source_bias.py
python scripts/02_vifn_experiments/10_learning_curve_vifn.py
~~~


## Phụ lục B — Kiểm tra đồng bộ


In [ ]:
checks = {
    "clean train samples": len(clean_train) == 924,
    "official test samples": len(raw_test) == 282,
    "test checksum": test_sha256 == final_metadata["test_sha256"],
    "model C": svm.C == 1.0,
    "model class weight": svm.class_weight == "balanced",
    "best feature": best_feature["experiment"] == "M03_word_bigram",
    "best tuning rank": best_cv["rank_test_macro_f1"] == 1,
    "recorded vs recomputed Test": metric_comparison["absolute_difference"].max() < 1e-12,
    "official Test Macro F1": np.isclose(official_recomputed["macro_f1"], 0.8788231130883171),
    "error count": len(error_analysis) == 34,
    "interpretability count": len(error_explanations) == 34,
}
check_table = pd.Series(checks, name="passed").to_frame()
display(check_table)
assert check_table["passed"].all()
print("OK — ViFN notebook đồng bộ với dữ liệu, model và report đã đóng băng.")
